In [1]:
pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [3]:
# =============================================================================
# 1. 라이브러리 임포트 및 설정
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

# 모델링 라이브러리
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    classification_report, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, IsolationForest

# 부스팅 모델 (설치 필요: pip install xgboost lightgbm)
import xgboost as xgb
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")

print("라이브러리 로드 완료. 5950X 엔진 가동 준비 끝.")

# =============================================================================
# 2. 데이터 로드 및 기본 전처리 (대표님의 핵심 로직)
# =============================================================================
# 파일 경로가 맞는지 확인해주세요
df = pd.read_csv("creditcard.csv")

def base_feature_engineering(data: pd.DataFrame) -> pd.DataFrame:
    d = data.copy()

    # Time 파생 (시간의 주기성 반영)
    d['hour'] = (d['Time'] // 3600) % 24
    d['hour_sin'] = np.sin(2 * np.pi * d['hour'] / 24)
    d['hour_cos'] = np.cos(2 * np.pi * d['hour'] / 24)

    # Amount 파생 (로그 변환으로 분포 완화)
    d['Amount_log'] = np.log1p(d['Amount'])

    # ★★★ [대표님의 필살기] 상호작용 변수 ★★★
    # 사기 거래에서 동시에 튀는 변수들을 증폭시킴
    d['V17_V14'] = d['V17'] * d['V14']
    d['V12_V10'] = d['V12'] * d['V10']

    # Amount 스케일링 (RobustScaler로 이상치 영향 최소화)
    scaler = RobustScaler()
    d['Amount_scaled'] = scaler.fit_transform(d[['Amount']])

    # 불필요 원본 컬럼 제거
    d = d.drop(['Time', 'Amount', 'hour'], axis=1)

    return d

# 전처리 적용
df_fe = base_feature_engineering(df)

X = df_fe.drop('Class', axis=1)
y = df_fe['Class']

# 학습/테스트 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"데이터 준비 완료. Train: {X_train.shape}, Test: {X_test.shape}")

# =============================================================================
# 3. 고급 전처리: IsolationForest 점수 추가 (AI도 감탄한 부분)
# =============================================================================
def add_iforest_score_full(X_tr, X_te, random_state=42):
    # V로 시작하는 변수만 사용
    v_cols = [c for c in X_tr.columns if c.startswith('V')]
    
    # IsolationForest 학습 (Train에만 fit하여 누수 방지)
    iso = IsolationForest(contamination=0.002, random_state=random_state, n_jobs=-1)
    iso.fit(X_tr[v_cols])

    X_tr2 = X_tr.copy()
    X_te2 = X_te.copy()

    # "얼마나 이상한가?" 점수를 피처로 추가
    X_tr2["anomaly_score"] = iso.decision_function(X_tr[v_cols])
    X_te2["anomaly_score"] = iso.decision_function(X_te[v_cols])

    return X_tr2, X_te2

print("IsolationForest 점수 계산 중... (5950X라 금방 끝날 겁니다)")
X_train2, X_test2 = add_iforest_score_full(X_train, X_test)

# 컬럼 순서 동기화 (에러 방지용 안전장치)
X_test2 = X_test2.reindex(columns=X_train2.columns)

# =============================================================================
# 4. 모델링: 4대장 앙상블 (LR + RF + XGB + LightGBM)
# =============================================================================
# 불균형 데이터 비율 계산
ratio = float((y_train == 0).sum()) / (y_train == 1).sum()

print("모델 4대장 소환 중...")

# 1) Logistic Regression (기본기 담당)
lr = LogisticRegression(max_iter=4000, class_weight="balanced", n_jobs=-1)

# 2) RandomForest (안정성 담당 - 5950X 믿고 트리 개수 500개로 상향)
rf = RandomForestClassifier(
    n_estimators=500, 
    random_state=42, 
    n_jobs=-1, 
    class_weight="balanced_subsample"
)

# 3) XGBoost (성능 담당)
xgb_model = xgb.XGBClassifier(
    n_estimators=2000,
    max_depth=6,            # 깊이 약간 증가
    learning_rate=0.02,     # 학습률을 낮춰서 더 꼼꼼하게
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio, # 불균형 해결
    tree_method="hist",     # 속도 최적화
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

# 4) ★ LightGBM (NEW: 속도와 성능의 밸런스, 친구 추가)
lgbm_model = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=64,          # 5950X니까 좀 더 복잡하게
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio,
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

# 5) 앙상블 (투표 시스템)
voting = VotingClassifier(
    estimators=[
        ("lr", lr), 
        ("rf", rf), 
        ("xgb", xgb_model), 
        ("lgbm", lgbm_model) # LightGBM 합류
    ],
    voting="soft",
    n_jobs=-1
)

print("앙상블 학습 시작! (CPU가 화르륵 일할 겁니다)")
voting.fit(X_train2, y_train)

# =============================================================================
# 5. 평가 및 결과 확인
# =============================================================================
# 예측
vote_prob = voting.predict_proba(X_test2)[:, 1]

# 성능 측정
roc = roc_auc_score(y_test, vote_prob)
ap = average_precision_score(y_test, vote_prob)

# 최적의 Threshold 찾기 (F1 Score 기준)
precision, recall, thresholds = precision_recall_curve(y_test, vote_prob)
f1_scores = 2 * recall * precision / (recall + precision + 1e-12)
best_idx = np.argmax(f1_scores)
best_thresh = thresholds[best_idx]
best_f1 = f1_scores[best_idx]

vote_pred = (vote_prob >= best_thresh).astype(int)

print("\n" + "="*40)
print(f" [최종 성적표] - 4Model Ensemble")
print("="*40)
print(f" ROC-AUC : {roc:.4f}")
print(f" PR-AUC  : {ap:.4f} (가장 중요한 지표)")
print(f" Best F1 : {best_f1:.4f} (Threshold: {best_thresh:.4f})")
print("-" * 40)
print("Confusion Matrix:")
print(confusion_matrix(y_test, vote_pred))
print("\nClassification Report:")
print(classification_report(y_test, vote_pred, digits=4))

# test.csv와 sample_submission.csv가 같은 폴더에 있어야 합니다
test_df = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

id_col = sample.columns[0]
target_col = [c for c in sample.columns if c != id_col][0]

# 테스트 데이터에도 똑같은 전처리 적용
print("테스트 데이터 전처리 중...")
test_fe = base_feature_engineering(test_df)
_, test_fe2 = add_iforest_score_full(X_train, test_fe) # Train 기준으로 점수 매김

# 컬럼 순서 맞추기 (중요)
test_fe2 = test_fe2.reindex(columns=X_train2.columns)

# 예측 (아까 학습한 4대장 voting 모델 사용)
print("예측 수행 중...")
test_prob = voting.predict_proba(test_fe2)[:, 1]

# 최적의 Threshold로 0/1 분류
test_pred = (test_prob >= best_thresh).astype(int)

# 제출 파일 만들기
submission = pd.DataFrame({id_col: test_df[id_col], target_col: test_pred})
submission.to_csv("submission_5950X_Ensemble.csv", index=False)

print(f"제출 파일 저장 완료: submission_5950X_Ensemble.csv")
print(submission.head())

라이브러리 로드 완료. 5950X 엔진 가동 준비 끝.
데이터 준비 완료. Train: (227845, 34), Test: (56962, 34)
IsolationForest 점수 계산 중... (5950X라 금방 끝날 겁니다)
모델 4대장 소환 중...
앙상블 학습 시작! (CPU가 화르륵 일할 겁니다)

 [최종 성적표] - 4Model Ensemble
 ROC-AUC : 0.9740
 PR-AUC  : 0.8685 (가장 중요한 지표)
 Best F1 : 0.8791 (Threshold: 0.7996)
----------------------------------------
Confusion Matrix:
[[56860     4]
 [   18    80]]

Classification Report:
              precision    recall  f1-score   support

           0     0.9997    0.9999    0.9998     56864
           1     0.9524    0.8163    0.8791        98

    accuracy                         0.9996     56962
   macro avg     0.9760    0.9081    0.9395     56962
weighted avg     0.9996    0.9996    0.9996     56962

테스트 데이터 전처리 중...
예측 수행 중...
제출 파일 저장 완료: submission_5950X_Ensemble.csv
       id  Class
0  170883      0
1  170884      0
2  170885      0
3  170886      0
4  170887      0
